In [13]:
import pandas as pd
import numpy as np
import re
#import nb_black

johor = pd.read_csv("johor_prop.csv")
johor.head()

# show full row
pd.options.display.max_columns = None

In [14]:
# remove unwanted column

johor = johor[['House Type', 'Price', 'Location', 'Size (sq.ft)',
       'No.of Bed', 'No.of Bath', 'Land Status']]
print(johor.head())
print(johor.shape)

                                          House Type       Price  \
0   New Terraced House  for sale, listed 2 weeks ago  RM 717,000   
1  New 1-storey Terraced House  for sale, listed ...  RM 288,000   
2  New 2-storey Terraced House  for sale, listed ...  RM 365,000   
3  New 2-storey Terraced House  for sale, listed ...  RM 680,000   
4  New Service Residence  for sale, listed 15 hou...  RM 398,000   

                 Location Size (sq.ft)  No.of Bed  No.of Bath Land Status  
0      Johor Bahru, Johor        1,823        4.0         3.0    Freehold  
1            Kulai, Johor        1,300        3.0         2.0    Freehold  
2       Ayer Hitam, Johor        1,400        4.0         3.0   Leasehold  
3  Iskandar Puteri, Johor        1,500        4.0         3.0    Freehold  
4      Johor Bahru, Johor          510        1.0         1.0    Freehold  
(265, 7)


In [15]:
# renaming columns name for standardization

johor.rename(columns= {
    'House Type': 'property_type',
    'Price': 'price', 
    'Location': 'location', 
    'Size (sq.ft)': 'size_sqft', 
    'No.of Bed': 'total_bedroom',
    'No.of Bath': 'total_bathroom', 
    'Land Status': 'land_status'
}, inplace = True)

johor.columns


Index(['property_type', 'price', 'location', 'size_sqft', 'total_bedroom',
       'total_bathroom', 'land_status'],
      dtype='object')

In [16]:
# cleaning each column before proceed to Statistical Analysis

# 1. property_type

johor["property_type"].head()

house_types = []

regex_property = r"(?<=New\s)(\S+)"


for house in johor["property_type"]:
    house_matches = re.search(regex_property, house)
    if house_matches is not None:
        house_type = house_matches.group(0).strip()
    else:
        house_type = ""
    house_types.append(house_type)
    
#print(house_types)

johor["property_type_extract"] = house_types
print(johor.head())

                                       property_type       price  \
0   New Terraced House  for sale, listed 2 weeks ago  RM 717,000   
1  New 1-storey Terraced House  for sale, listed ...  RM 288,000   
2  New 2-storey Terraced House  for sale, listed ...  RM 365,000   
3  New 2-storey Terraced House  for sale, listed ...  RM 680,000   
4  New Service Residence  for sale, listed 15 hou...  RM 398,000   

                 location size_sqft  total_bedroom  total_bathroom  \
0      Johor Bahru, Johor     1,823            4.0             3.0   
1            Kulai, Johor     1,300            3.0             2.0   
2       Ayer Hitam, Johor     1,400            4.0             3.0   
3  Iskandar Puteri, Johor     1,500            4.0             3.0   
4      Johor Bahru, Johor       510            1.0             1.0   

  land_status property_type_extract  
0    Freehold              Terraced  
1    Freehold              1-storey  
2   Leasehold              2-storey  
3    Freehold     

In [17]:
# 2. price

johor["prices"] = johor["price"].str.split(expand=True)[1].str.replace(",","")
johor["prices"] = johor["prices"].astype("int")
johor["prices"].dtype

dtype('int32')

In [18]:
# 3. location - since all properties in the dataset are from Johor, so we wil focus on the district only

# lower case each letter
johor["location"] = [x.lower() for x in johor["location"]]
# Check for no. of district under Johor < suppose total under 
#print(johor["location"].value_counts())  # total 58

# check the value of the unknown district
#print(johor["location"].nunique())

# Handling inconsistent lcoation names

# info from google - Batu Pahat, Johor Bahru, Kluang, Kota Tinggi, Mersing, Muar, Pontian, and Segamat (8 district)

valid_districts = {
    "batu pahat": "batu pahat",
    "johor bahru": "johor bahru",
    "kluang": "kluang",
    "kota tinggi": "kota tinggi",
    "mersing": "mersing",
    "muar": "muar",
    "pontian": "pontian",
    "segamat": "segamat"
}

town_to_district = {
    "johor bahru": "johor bahru",
    "kulai": "kulai",
    "kluang": "kluang",
    "pasir gudang": "johor bahru",
    "senai": "kulai",
    "skudai": "johor bahru",
    "iskandar puteri": "johor bahru",
    "permas jaya": "johor bahru",
    "masai": "johor bahru",
    "muar": "johor",
    "gelang patah": "johor bahru",
    "parkland by the river": "johor bahru",
    "tebrau": "johor bahru",
    "sentrio residences @ senai": "kulai",
    "horizon hills": "johor bahru",
    "kota tinggi": "kota tinggi",
    "pangsapuri ksl bukit gemilang": "johor bahru",
    "tangkak": "muar",
    "ayer hitam": "kluang",
    "d' secret garden": "johor bahru",
    "pengerang": "kota tinggi",
    "bandar baru permas jaya": "johor bahru",
    "parc regency": "johor bahru",
    "ksl residence 2 @ kangkar tebrau": "johor bahru",
    "veranda residence": "johor bahru",
    "setia indah": "johor bahru",
    "d'secret garden @ kempas indah": "johor bahru",
    "verte medini condominium": "johor bahru",
    "desaru utama residence": "kota tinggi",
    "the senai garden": "kulai",
    "batu pahat": "batu pahat",
    "pontian": "pontian",
    "pandan residence": "johor bahru",
    "bakri": "muar",
    "mersing": "mersing",
    "sierra heights (residensi siera perdana)": "johor bahru",
    "yong peng": "kluang",
    "puteri harbour": "johor bahru",
    "santai @ eco spring": "johor bahru",
    "mutiara austin": "johor bahru",
    "east bay (seri bayan)": "johor bahru",
    "the garden residences": "johor bahru",
    "senibong": "johor bahru",
    "m minori": "johor bahru",
    "d'summit residences": "johor bahru",
    "seri austin residence luxury apartment": "johor bahru",
    "marina cove": "johor bahru",
    "setia tropika": "johor bahru",
    "permas sentral": "johor bahru",
    "idaman residence @ nusa idaman": "johor bahru",
    "tampoi height serviced apartment": "johor bahru",
    "iskandar residences medini": "johor bahru",
    "vista tiara @ mbw bay": "johor bahru",
    "arc @ austin hills": "johor bahru",
    "ksl residences @ daya": "johor bahru",
    "aliva @ mount austin": "johor bahru",
    "kings bay @ country garden danga bay": "johor bahru",
    "ulu tiram": "johor bahru"
}

# function to get the correct district

def get_district(location):
    parts = location.split(",")
    for part in parts:
        part = part.strip()
        if part in valid_districts:
            return valid_districts[part]
        elif part in town_to_district:
            return town_to_district[part]
    else:
        return "Unknown"
        
johor["district"] = johor["location"].apply(get_district)
print(johor["district"].value_counts())

district
johor bahru    164
kulai           45
kluang          29
kota tinggi     11
muar            11
batu pahat       2
pontian          2
mersing          1
Name: count, dtype: int64


In [19]:
johor.head()

,property_type,price,location,size_sqft,total_bedroom,total_bathroom,land_status,property_type_extract,prices,district
0,"New Terraced House for sale, listed 2 weeks ago","RM 717,000","johor bahru, johor","1,823",4.0,3.0,Freehold,Terraced,717000,johor bahru
1,"New 1-storey Terraced House for sale, listed ...","RM 288,000","kulai, johor","1,300",3.0,2.0,Freehold,1-storey,288000,kulai
2,"New 2-storey Terraced House for sale, listed ...","RM 365,000","ayer hitam, johor","1,400",4.0,3.0,Leasehold,2-storey,365000,kluang
3,"New 2-storey Terraced House for sale, listed ...","RM 680,000","iskandar puteri, johor","1,500",4.0,3.0,Freehold,2-storey,680000,johor bahru
4,"New Service Residence for sale, listed 15 hou...","RM 398,000","johor bahru, johor",510,1.0,1.0,Freehold,Service,398000,johor bahru


In [20]:
# size_sqft

johor["size_sqft"] = johor["size_sqft"].str.replace(",", "").astype("int")
johor["size_sqft"].dtype



dtype('int32')

In [21]:
# property_type_extract

valid_property_type = {
    "landed": "landed",
    "high_rise": "high_rise",
    "shop_lot": "shop_lot",
    "factory": "factory"
}

property_type = {
    "Service": "high_rise",
    "2-storey": "landed",
    "1-storey": "landed",
    "Apartment": "high_rise",
    "Terraced": "landed",
    "Semi-Detached": "landed",
    "Condominium": "high_rise",
    "Cluster": "landed",
    "Shop": "shop_lot",
    "Others": "shop_lot",
    "Warehouse": "factory",
    "Bungalow": "landed",
    "3-storey": "high_rise",
    "2.5-storey": "high_rise",
    "Office": "shop_lot",
    "Flat": "high_rise"
}

def get_type(property_name):
    return property_type.get(property_name, "Unknown")

johor["property_type_extract"] = johor["property_type_extract"].apply(get_type)
johor["property_type_extract"].value_counts()

property_type_extract
landed       132
high_rise    114
shop_lot      13
factory        6
Name: count, dtype: int64

In [22]:
# Finalize all columns

# remove unwanted columns
johor = johor[["district", "property_type_extract", "size_sqft", "total_bedroom", "total_bathroom", "land_status", "prices"]]
johor.rename(columns={
    "property_type_extract": "property_type",
    "district": "location"}, inplace=True)
johor.head()

,location,property_type,size_sqft,total_bedroom,total_bathroom,land_status,prices
0,johor bahru,landed,1823,4.0,3.0,Freehold,717000
1,kulai,landed,1300,3.0,2.0,Freehold,288000
2,kluang,landed,1400,4.0,3.0,Leasehold,365000
3,johor bahru,landed,1500,4.0,3.0,Freehold,680000
4,johor bahru,high_rise,510,1.0,1.0,Freehold,398000


In [23]:
johor.to_csv(r"C:\Users\User\Desktop\Data Analyst\End To End Project\Project 8 - Johor Property\transform_johor_prop.csv")